# Video Question Answering with Temporal Grounding

A bilingual (English / Arabic) multimodal video QA pipeline. Given a video and a natural-language question, the system retrieves the most relevant moments using fused visual (SigLIP) and audio-transcript (Whisper + bge-m3) signals, generates a grounded answer with cited timestamps via a Llama 3 LLM, and produces four explainability views: a relevance timeline, a Grad-CAM saliency map on the top frame, a candidate-frame grid, and a modality-contribution chart.

**Run on Google Colab (recommended):**

1. **Runtime → Change runtime type → T4 GPU.**
2. Add your Groq API key as a Colab Secret named `GROQ_API_KEY` (key icon in the left sidebar). Get a free key at <https://console.groq.com/keys>.
3. **Runtime → Run all.** First run downloads ~3 GB of model weights.
4. The final cell launches a Gradio app and prints a public URL valid for 72 hours.

## 1. Install dependencies

`ffmpeg` is installed via `apt`; the Python stack via `pip`. This only needs to run once per Colab session.

In [ ]:
!pip install -q faster-whisper transformers accelerate sentence-transformers \
    FlagEmbedding grad-cam gradio yt-dlp moviepy \
    opencv-python-headless python-dotenv groq matplotlib seaborn pandas
!apt-get install -y ffmpeg > /dev/null

## 2. Imports and GPU detection

In [ ]:
from __future__ import annotations

import os
import re
import math
import hashlib
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns  # noqa: F401
from tqdm.auto import tqdm

from transformers import AutoModel, AutoProcessor
from sentence_transformers import SentenceTransformer
from faster_whisper import WhisperModel

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

import yt_dlp
from groq import Groq
import gradio as gr

DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

if DEVICE == "cuda":
    name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"[gpu] {name} ({vram:.1f} GB VRAM)")
else:
    print("[gpu] no CUDA device — pipeline will run very slowly on CPU")

print(f"[env] torch {torch.__version__}, device={DEVICE}")

## 3. Configuration

The pipeline auto-selects encoder sizes based on available VRAM:
* `high` (>= 14 GB): SigLIP-large @ 384, Whisper large-v3
* `low` (< 14 GB or CPU): SigLIP-base @ 224, Whisper medium

The Groq API key is read from Colab Secrets, then env vars / `.env`.

In [ ]:
def detect_compute_tier() -> str:
    """Return ``"high"`` if >= 14 GB of CUDA VRAM is available, else ``"low"``."""
    if not torch.cuda.is_available():
        return "low"
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    return "high" if vram_gb >= 14 else "low"


TIER = detect_compute_tier()

VISUAL_MODEL = {
    "high": "google/siglip-large-patch16-384",
    "low": "google/siglip-base-patch16-224",
}[TIER]
ASR_MODEL = "large-v3" if TIER == "high" else "medium"


@dataclass
class Config:
    # Sampling
    FPS_SAMPLE: float = 2.0
    SCENE_CHANGE_THRESHOLD: float = 0.85
    TRANSCRIPT_CHUNK_SECONDS: int = 10

    # Retrieval
    TOP_K_MOMENTS: int = 5
    VISUAL_WEIGHT: float = 0.5
    AUDIO_WEIGHT: float = 0.5
    RETRIEVAL_WINDOW_S: float = 1.0
    SCORE_TEMPERATURE: float = 0.05
    HIT_TOLERANCE_S: float = 3.0

    # Models
    VISUAL_MODEL: str = VISUAL_MODEL
    WHISPER_MODEL: str = ASR_MODEL
    TEXT_RETRIEVAL_MODEL: str = "BAAI/bge-m3"
    GROQ_MODEL: str = "llama-3.3-70b-versatile"

    WORK_DIR: str = "/content"


CONFIG = Config()

vram_str = (
    f"{torch.cuda.get_device_properties(0).total_memory / (1024 ** 3):.1f} GB VRAM"
    if torch.cuda.is_available()
    else "CPU"
)
print(f"[config] compute tier: {TIER} ({vram_str})")
print(f"[config] visual encoder: {CONFIG.VISUAL_MODEL}")
print(f"[config] ASR model: {CONFIG.WHISPER_MODEL}")
print(f"[config] frame sampling: {CONFIG.FPS_SAMPLE} fps, scene threshold: {CONFIG.SCENE_CHANGE_THRESHOLD}")


def load_groq_client() -> Groq:
    """Construct a Groq client from Colab Secrets, env vars, or `.env`."""
    key: str | None = None
    try:
        from google.colab import userdata  # type: ignore
        key = userdata.get("GROQ_API_KEY")
    except Exception:
        pass
    if not key:
        try:
            from dotenv import load_dotenv
            load_dotenv()
        except Exception:
            pass
        key = os.getenv("GROQ_API_KEY")
    if not key:
        raise RuntimeError(
            "GROQ_API_KEY is not set. On Colab add it via the Secrets panel; "
            "locally put it in a .env file or export it."
        )
    return Groq(api_key=key)


GROQ_CLIENT = load_groq_client()
print("[config] Groq client ready")

## 4. Load models

Each model loads once and is cached. First-run downloads of the high-tier stack total ~3 GB (~3 minutes on Colab); subsequent runs reuse the Hugging Face cache.

In [ ]:
_MODEL_CACHE: dict[str, Any] = {}


def get_visual_encoder() -> tuple[Any, Any]:
    """Return ``(model, processor)`` for the configured SigLIP variant."""
    if "siglip" not in _MODEL_CACHE:
        model = AutoModel.from_pretrained(CONFIG.VISUAL_MODEL).to(DEVICE).eval()
        processor = AutoProcessor.from_pretrained(CONFIG.VISUAL_MODEL)
        _MODEL_CACHE["siglip"] = (model, processor)
        print(f"[load] visual encoder: {CONFIG.VISUAL_MODEL}")
    return _MODEL_CACHE["siglip"]


def get_text_retriever() -> SentenceTransformer:
    """Return a cached bge-m3 sentence-transformer."""
    if "bge" not in _MODEL_CACHE:
        _MODEL_CACHE["bge"] = SentenceTransformer(CONFIG.TEXT_RETRIEVAL_MODEL, device=DEVICE)
        print(f"[load] text retriever: {CONFIG.TEXT_RETRIEVAL_MODEL}")
    return _MODEL_CACHE["bge"]


def get_whisper() -> WhisperModel:
    """Return a cached faster-whisper model."""
    if "whisper" not in _MODEL_CACHE:
        ctype = "float16" if DEVICE == "cuda" else "int8"
        _MODEL_CACHE["whisper"] = WhisperModel(
            CONFIG.WHISPER_MODEL, device=DEVICE, compute_type=ctype,
        )
        print(f"[load] ASR: {CONFIG.WHISPER_MODEL} ({ctype})")
    return _MODEL_CACHE["whisper"]


# Eager-load to keep first user query fast.
_ = get_visual_encoder()
_ = get_text_retriever()
_ = get_whisper()
print("[load] all models ready")

## 5. Video ingestion

Accepts either a local path or an HTTP(S) URL. URL downloads use yt-dlp with an
Android/iOS player-client fallback for YouTube; data-center IPs (including
Colab) often fail with the default web client. URL downloads on Instagram and
similar gated hosts will fail without authentication — direct file upload is
the reliable path.

In [ ]:
class VideoDownloadError(RuntimeError):
    """Raised when a remote video URL cannot be fetched (login required, IP block, etc.)."""


def is_url(s: str) -> bool:
    return bool(re.match(r"^https?://", (s or "").strip()))


def download_video(source: str) -> str:
    """Resolve ``source`` to a local file path.

    Args:
        source: Either an absolute/relative path or an HTTP(S) URL.

    Raises:
        ValueError: ``source`` is empty.
        FileNotFoundError: Local path does not exist.
        VideoDownloadError: yt-dlp could not fetch the URL.
    """
    if not source:
        raise ValueError("Empty video source.")

    if not is_url(source):
        path = Path(source).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f"Video file not found: {path}")
        return str(path)

    Path(CONFIG.WORK_DIR).mkdir(parents=True, exist_ok=True)
    out_template = str(Path(CONFIG.WORK_DIR) / "downloaded_video.%(ext)s")
    ydl_opts = {
        "outtmpl": out_template,
        "format": "mp4/bestvideo[ext=mp4]+bestaudio[ext=m4a]/best",
        "quiet": True,
        "no_warnings": True,
        "noplaylist": True,
        "merge_output_format": "mp4",
        "extractor_args": {"youtube": {"player_client": ["android", "ios", "web"]}},
    }
    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(source, download=True)
            filename = ydl.prepare_filename(info)
    except Exception as e:
        host = re.sub(r"^https?://(www\.)?", "", source).split("/")[0]
        raise VideoDownloadError(
            f"Could not download from {host}. The URL is probably login-gated "
            f"or the host is blocking Colab's IP range. Upload the video file "
            f"directly via the Inputs tab instead.\n\nUnderlying error: {e}"
        ) from e

    return str(Path(filename).resolve())

## 6. Frame extraction with keyframe deduplication

Sample at `FPS_SAMPLE` frames per second, then drop any frame whose normalized
3-channel histogram correlates above `SCENE_CHANGE_THRESHOLD` with its
predecessor. This keeps per-second cost roughly constant when raising the
sampling rate, since static scenes (e.g. a held shot) collapse to one
representative frame.

In [ ]:
def extract_frames(
    video_path: str,
    fps: float,
    scene_threshold: float,
) -> list[tuple[float, np.ndarray]]:
    """Return ``[(timestamp_s, rgb_frame), ...]`` after keyframe deduplication."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f"Cannot open video: {video_path}")

    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    step = max(int(round(src_fps / max(fps, 0.1))), 1)

    frames: list[tuple[float, np.ndarray]] = []
    last_hist: np.ndarray | None = None
    idx = 0
    pbar = tqdm(total=total, desc="Extracting frames", leave=False)
    while True:
        ret, frame_bgr = cap.read()
        if not ret:
            break
        if idx % step == 0:
            hist = cv2.calcHist(
                [frame_bgr], [0, 1, 2], None, [8, 8, 8], [0, 256, 0, 256, 0, 256],
            )
            cv2.normalize(hist, hist)
            if last_hist is None or cv2.compareHist(last_hist, hist, cv2.HISTCMP_CORREL) < scene_threshold:
                ts = idx / src_fps
                frames.append((ts, cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)))
                last_hist = hist
        idx += 1
        pbar.update(1)
    pbar.close()
    cap.release()
    return frames


def get_video_duration(video_path: str) -> float:
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    cap.release()
    return n / fps if fps > 0 else 0.0

## 7. Speech transcription

`faster-whisper` auto-detects the language per chunk and returns word-level
timestamps that are later used to align retrieval windows.

In [ ]:
def transcribe_audio(video_path: str) -> list[dict]:
    """Return ``[{"start", "end", "text", "language"}, ...]`` for the video's audio.

    Returns an empty list if the audio track is silent or transcription fails.
    """
    model = get_whisper()
    try:
        segments_iter, info = model.transcribe(
            video_path, beam_size=5, vad_filter=True, word_timestamps=True,
        )
    except Exception as e:
        print(f"[asr] failed: {e}")
        return []

    lang = getattr(info, "language", "unknown") if info else "unknown"
    out: list[dict] = []
    for seg in tqdm(segments_iter, desc="Transcribing", leave=False):
        text = (seg.text or "").strip()
        if not text:
            continue
        out.append({
            "start": float(seg.start),
            "end": float(seg.end),
            "text": text,
            "language": lang,
        })
    return out

## 8. Multimodal indexing

Three small helpers: SigLIP image embeddings, transcript chunking by sliding
window, and bge-m3 transcript embeddings. All embeddings are L2-normalized so
cosine similarity reduces to a dot product.

In [ ]:
def _to_tensor(out: Any) -> torch.Tensor:
    """Coerce a model forward output to a tensor across transformers versions."""
    if isinstance(out, torch.Tensor):
        return out
    for attr in ("image_embeds", "text_embeds", "pooler_output", "last_hidden_state"):
        v = getattr(out, attr, None)
        if isinstance(v, torch.Tensor):
            return v
    raise TypeError(f"Cannot extract tensor from {type(out).__name__}")


@torch.no_grad()
def embed_frames(frames: list[tuple[float, np.ndarray]], batch_size: int = 16) -> torch.Tensor:
    """L2-normalized SigLIP image embeddings for every frame."""
    if not frames:
        return torch.empty(0, 768, device=DEVICE)
    model, processor = get_visual_encoder()
    feats: list[torch.Tensor] = []
    for i in tqdm(range(0, len(frames), batch_size), desc="Embedding frames", leave=False):
        batch = [Image.fromarray(f) for _, f in frames[i:i + batch_size]]
        inputs = processor(images=batch, return_tensors="pt").to(DEVICE)
        out = _to_tensor(model.get_image_features(**inputs))
        feats.append(F.normalize(out, dim=-1))
    return torch.cat(feats, dim=0)


def chunk_transcript(segments: list[dict], window_s: int) -> list[dict]:
    """Group consecutive segments into ~``window_s``-second chunks."""
    if not segments:
        return []
    chunks: list[dict] = []
    cur_start = segments[0]["start"]
    cur_end = cur_start + window_s
    cur_texts: list[str] = []
    cur_lang = segments[0].get("language", "unknown")

    for seg in segments:
        if seg["start"] >= cur_end and cur_texts:
            chunks.append({
                "start": cur_start,
                "end": min(cur_end, seg["start"]),
                "text": " ".join(cur_texts).strip(),
                "language": cur_lang,
            })
            cur_start = seg["start"]
            cur_end = cur_start + window_s
            cur_texts = []
        cur_texts.append(seg["text"])

    if cur_texts:
        chunks.append({
            "start": cur_start,
            "end": cur_end,
            "text": " ".join(cur_texts).strip(),
            "language": cur_lang,
        })
    return chunks


@torch.no_grad()
def embed_text_chunks(chunks: list[dict]) -> torch.Tensor:
    """L2-normalized bge-m3 embeddings of transcript chunks."""
    if not chunks:
        return torch.empty(0, 1024, device=DEVICE)
    model = get_text_retriever()
    embs = model.encode(
        [c["text"] for c in chunks],
        convert_to_tensor=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    )
    return embs.to(DEVICE)

## 9. Retrieval

Per timestamp the system computes a raw visual score (best similarity within
`±RETRIEVAL_WINDOW_S` of the timestamp) and a raw audio score (best similarity
over chunks containing the timestamp). Each modality's raw scores are then
**normalized via temperature-scaled z-softmax**, which sharpens peaks and lets
the linearly-fused combined score actually discriminate between candidates.
The top-K combined moments are returned, deduplicated so they do not overlap.

In [ ]:
_ARABIC = re.compile(r"[\u0600-\u06FF]")
_TRANSLATION_CACHE: dict[str, str] = {}


def detect_question_language(text: str) -> str:
    """Return ``"ar"`` if any Arabic codepoint appears in ``text``, else ``"en"``."""
    return "ar" if _ARABIC.search(text or "") else "en"


def translate_to_english(text: str) -> str:
    """Translate ``text`` to English via Groq, with an in-memory cache."""
    if not text or detect_question_language(text) == "en":
        return text
    if text in _TRANSLATION_CACHE:
        return _TRANSLATION_CACHE[text]
    try:
        resp = GROQ_CLIENT.chat.completions.create(
            model=CONFIG.GROQ_MODEL,
            messages=[
                {"role": "system",
                 "content": "Translate the user text to English. Output ONLY the translation, no explanation."},
                {"role": "user", "content": text},
            ],
            temperature=0.0,
            max_tokens=128,
        )
        en = (resp.choices[0].message.content or text).strip()
    except Exception:
        en = text
    _TRANSLATION_CACHE[text] = en
    return en


def normalize_scores(scores: np.ndarray, temperature: float) -> np.ndarray:
    """Z-score then temperature-softmax. Sharp distribution that highlights peaks."""
    if scores.size == 0:
        return scores
    std = scores.std()
    if std < 1e-8:
        return np.full_like(scores, 1.0 / scores.size)
    z = (scores - scores.mean()) / (std + 1e-8)
    e = np.exp(z / max(temperature, 1e-6))
    s = e.sum()
    return e / s if s > 0 else e


@torch.no_grad()
def _embed_text_visual(text: str) -> torch.Tensor:
    """Encode ``text`` with the SigLIP text tower; returns an L2-normalized (1, D) tensor."""
    model, processor = get_visual_encoder()
    inputs = processor(
        text=[text], return_tensors="pt", padding="max_length", truncation=True,
    ).to(DEVICE)
    out = _to_tensor(model.get_text_features(**inputs))
    return F.normalize(out, dim=-1)


@torch.no_grad()
def _embed_text_bge(text: str) -> torch.Tensor:
    model = get_text_retriever()
    return model.encode(
        [text], convert_to_tensor=True, normalize_embeddings=True,
    ).to(DEVICE)


def _audio_score_at(t: float, chunk_sims: np.ndarray, chunks: list[dict]) -> float:
    if chunk_sims.size == 0:
        return 0.0
    best = 0.0
    for i, ch in enumerate(chunks):
        if ch["start"] <= t <= ch["end"]:
            best = max(best, float(chunk_sims[i]))
    return best


def _visual_score_at(
    t: float,
    frame_sims: np.ndarray,
    frames: list[tuple[float, np.ndarray]],
    window_s: float,
) -> tuple[float, int]:
    if frame_sims.size == 0:
        return 0.0, -1
    best, best_idx = 0.0, -1
    for i, (ts, _) in enumerate(frames):
        if abs(ts - t) <= window_s:
            s = float(frame_sims[i])
            if s > best:
                best, best_idx = s, i
    return best, best_idx


def retrieve_moments(
    question: str,
    question_en: str,
    frames: list[tuple[float, np.ndarray]],
    frame_embeds: torch.Tensor,
    chunks: list[dict],
    chunk_embeds: torch.Tensor,
    top_k: int,
    visual_weight: float,
    audio_weight: float,
    window_s: float,
    temperature: float,
) -> list[dict]:
    """Return the top-K most relevant moments, with both raw and normalized scores."""
    if not frames:
        return []

    q_visual = _embed_text_visual(question_en)
    q_text = _embed_text_bge(question)

    if frame_embeds.numel():
        frame_sims_raw = (frame_embeds @ q_visual.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
    else:
        frame_sims_raw = np.zeros(0)
    if chunk_embeds.numel():
        chunk_sims_raw = (chunk_embeds @ q_text.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
    else:
        chunk_sims_raw = np.zeros(0)

    n = len(frames)
    visual_raw = np.zeros(n)
    audio_raw = np.zeros(n)
    transcripts = [""] * n
    best_idx = list(range(n))

    for i, (ts, _) in enumerate(frames):
        v, vi = _visual_score_at(ts, frame_sims_raw, frames, window_s)
        visual_raw[i] = v
        if vi >= 0:
            best_idx[i] = vi
        audio_raw[i] = _audio_score_at(ts, chunk_sims_raw, chunks)
        for ch in chunks:
            if ch["start"] <= ts <= ch["end"]:
                transcripts[i] = ch["text"]
                break

    v_norm = normalize_scores(visual_raw, temperature)
    a_norm = normalize_scores(audio_raw, temperature)
    combined = visual_weight * v_norm + audio_weight * a_norm

    candidates: list[dict] = []
    for i, (ts, frame) in enumerate(frames):
        candidates.append({
            "timestamp": float(ts),
            "visual_score": float(visual_raw[i]),
            "audio_score": float(audio_raw[i]),
            "visual_score_norm": float(v_norm[i]) if v_norm.size else 0.0,
            "audio_score_norm": float(a_norm[i]) if a_norm.size else 0.0,
            "combined": float(combined[i]) if combined.size else 0.0,
            "frame": frames[best_idx[i]][1] if 0 <= best_idx[i] < n else frame,
            "transcript": transcripts[i],
            "frame_index": best_idx[i],
        })

    candidates.sort(key=lambda c: c["combined"], reverse=True)
    selected: list[dict] = []
    for cand in candidates:
        if all(abs(cand["timestamp"] - s["timestamp"]) > window_s * 2 for s in selected):
            selected.append(cand)
        if len(selected) >= top_k:
            break
    return selected

## 10. Answer generation

The Groq LLM (`llama-3.3-70b-versatile`) is asked to answer using only the
retrieved evidence and to reply in the same language as the question.

In [ ]:
def _format_seconds(t: float) -> str:
    m, s = divmod(int(round(t)), 60)
    return f"{m:02d}:{s:02d}"


def generate_answer(question: str, moments: list[dict], language: str) -> dict:
    """Return ``{"answer", "cited_timestamps"}`` for ``question`` over ``moments``."""
    if not moments:
        return {"answer": "No relevant moments were found in the video.",
                "cited_timestamps": []}

    evidence_lines: list[str] = []
    for m in moments:
        ts = _format_seconds(m["timestamp"])
        snippet = m["transcript"] or "(no speech in this window)"
        evidence_lines.append(
            f"- [{ts}] visual={m['visual_score']:.2f} audio={m['audio_score']:.2f} "
            f"transcript: {snippet}"
        )
    evidence = "\n".join(evidence_lines)

    reply_lang = "Arabic" if language == "ar" else "English"
    sys_msg = (
        "You answer questions about a video using ONLY the supplied evidence. "
        f"Reply in {reply_lang} with a short, precise answer and cite the timestamps "
        "your answer relies on in [mm:ss] format. If the evidence is insufficient, "
        "say so plainly."
    )
    user_msg = (
        f"Question: {question}\n\n"
        f"Evidence (top {len(moments)} moments, ranked):\n{evidence}\n\n"
        "Answer the question, citing timestamps."
    )

    try:
        resp = GROQ_CLIENT.chat.completions.create(
            model=CONFIG.GROQ_MODEL,
            messages=[
                {"role": "system", "content": sys_msg},
                {"role": "user", "content": user_msg},
            ],
            temperature=0.2,
            max_tokens=512,
        )
        answer = (resp.choices[0].message.content or "").strip()
    except Exception as e:
        answer = f"Groq API error: {e}\n\nRaw evidence:\n{evidence}"

    return {"answer": answer, "cited_timestamps": [m["timestamp"] for m in moments]}

## 11. XAI #1 — Timeline relevance

Plots raw per-timestamp visual and audio similarities. The mean is drawn as a
dotted reference line; the top-3 moments selected by the normalized combined
score are highlighted with vertical markers and their `mm:ss` labels.

In [ ]:
def plot_timeline(
    question: str,
    moments: list[dict],
    raw_visual: np.ndarray,
    raw_audio: np.ndarray,
    visual_ts: np.ndarray,
    audio_ts: np.ndarray,
    video_duration: float,
) -> plt.Figure:
    """Render the timeline relevance plot."""
    plt.style.use("seaborn-v0_8-whitegrid")
    fig, ax = plt.subplots(figsize=(11, 4))

    if visual_ts.size and raw_visual.size:
        ax.plot(visual_ts, raw_visual, color="#3b82f6", linewidth=1.5,
                label="Visual relevance (SigLIP)", alpha=0.95)
    if audio_ts.size and raw_audio.size:
        ax.plot(audio_ts, raw_audio, color="#f59e0b", linewidth=1.5,
                label="Audio relevance (bge-m3)", alpha=0.95)

    if raw_visual.size or raw_audio.size:
        all_scores = np.concatenate([s for s in (raw_visual, raw_audio) if s.size])
        ax.axhline(float(all_scores.mean()), color="black",
                   linestyle=":", linewidth=0.9, alpha=0.55, label="Mean")

    y_max = 1.0
    if raw_visual.size:
        y_max = max(y_max, float(raw_visual.max()) * 1.15)
    if raw_audio.size:
        y_max = max(y_max, float(raw_audio.max()) * 1.15)
    ax.set_ylim(0, y_max)
    ax.set_xlim(0, max(video_duration, 1.0))

    for m in moments[:3]:
        ax.axvline(m["timestamp"], color="#16a34a", linewidth=1.3, alpha=0.7)
        ax.text(
            m["timestamp"], y_max * 0.97, _format_seconds(m["timestamp"]),
            ha="center", va="top", fontsize=9, color="#15803d",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                      edgecolor="#16a34a", linewidth=0.7),
        )

    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Cosine similarity")
    ax.set_title(f"Timeline relevance — {question[:80]}")
    ax.legend(loc="upper right", framealpha=0.9)
    plt.tight_layout()
    return fig

## 12. XAI #2 — Grad-CAM on the top frame

Saliency overlay showing which patches of the top frame matched the question
embedding. Returns a status string describing success or the failure reason
so the UI can surface it cleanly.

In [ ]:
class _SimilarityWrapper(nn.Module):
    """Wrap a vision-language model so its forward returns a (B, 1) similarity score."""

    def __init__(self, model: nn.Module, text_features: torch.Tensor) -> None:
        super().__init__()
        self.model = model
        self.text_features = text_features  # (1, D), L2-normalized

    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        feats = _to_tensor(self.model.get_image_features(pixel_values=pixel_values))
        feats = F.normalize(feats, dim=-1)
        return feats @ self.text_features.T


def gradcam_on_frame(frame: np.ndarray, question_en: str) -> tuple[np.ndarray, str]:
    """Return ``(overlay_rgb, status_message)``. On failure returns the raw frame."""
    model, processor = get_visual_encoder()

    try:
        with torch.no_grad():
            text_inputs = processor(
                text=[question_en], return_tensors="pt",
                padding="max_length", truncation=True,
            ).to(DEVICE)
            text_features = _to_tensor(model.get_text_features(**text_inputs))
            text_features = F.normalize(text_features, dim=-1)

        wrapper = _SimilarityWrapper(model, text_features).to(DEVICE).eval()

        pil = Image.fromarray(frame)
        pixel_values = processor(images=pil, return_tensors="pt")["pixel_values"].to(DEVICE)
        pixel_values.requires_grad_(True)

        target_layers = [model.vision_model.encoder.layers[-1].layer_norm1]

        def reshape_transform(tensor: torch.Tensor) -> torch.Tensor:
            n = tensor.shape[1]
            side = int(round(math.sqrt(n)))
            if side * side != n:
                tensor = tensor[:, 1:, :]
                n = tensor.shape[1]
                side = int(round(math.sqrt(n)))
            return tensor.reshape(tensor.size(0), side, side, tensor.size(2)).permute(0, 3, 1, 2)

        cam = GradCAM(model=wrapper, target_layers=target_layers,
                      reshape_transform=reshape_transform)
        grayscale = cam(input_tensor=pixel_values, targets=[ClassifierOutputTarget(0)])[0]
    except Exception as e:
        return frame, f"Grad-CAM unavailable: {type(e).__name__}: {e}"

    img_resized = cv2.resize(frame, (grayscale.shape[1], grayscale.shape[0]))
    overlay = show_cam_on_image(img_resized.astype(np.float32) / 255.0,
                                 grayscale, use_rgb=True)
    return overlay, "Grad-CAM applied"

## 13. XAI #3 — Frame similarity grid

Top candidate frames in a 2x3 grid, each annotated with its raw and normalized
scores. The frame used in the final answer gets a green border.

In [ ]:
def plot_frame_similarity_grid(
    top_moments: list[dict],
    question: str,
    chosen_index: int = 0,
) -> plt.Figure:
    plt.style.use("seaborn-v0_8-whitegrid")
    n = min(len(top_moments), 6)
    if n == 0:
        fig, ax = plt.subplots(figsize=(11, 5))
        ax.text(0.5, 0.5, "No moments to display.", ha="center", va="center")
        ax.axis("off")
        return fig

    rows = 2 if n > 3 else 1
    cols = 3 if n > 1 else 1
    fig, axes = plt.subplots(rows, cols, figsize=(11, 5))
    axes = np.atleast_1d(axes).flatten()

    for i in range(rows * cols):
        ax = axes[i]
        if i < n:
            m = top_moments[i]
            ax.imshow(m["frame"])
            ax.set_title(
                f"#{i+1}  t={_format_seconds(m['timestamp'])}\n"
                f"V={m['visual_score']:.2f}  A={m['audio_score']:.2f}  "
                f"combined={m['combined']:.3f}",
                fontsize=10,
            )
            if i == chosen_index:
                for spine in ax.spines.values():
                    spine.set_edgecolor("#16a34a")
                    spine.set_linewidth(4)
            ax.set_xticks([])
            ax.set_yticks([])
        else:
            ax.axis("off")

    fig.suptitle(f"Top candidate moments — {question[:80]}", fontsize=12)
    plt.tight_layout()
    return fig

## 14. XAI #4 — Modality contribution

Stacked bars showing the **weighted, normalized** contribution of the visual
and audio modalities to each top moment's combined score. Answers "did the
model rely on what it saw or what it heard?" at a glance.

In [ ]:
def plot_modality_contribution(
    top_moments: list[dict],
    visual_weight: float,
    audio_weight: float,
) -> plt.Figure:
    plt.style.use("seaborn-v0_8-whitegrid")
    if not top_moments:
        fig, ax = plt.subplots(figsize=(9, 4))
        ax.text(0.5, 0.5, "No moments to display.", ha="center", va="center")
        ax.axis("off")
        return fig

    labels = [_format_seconds(m["timestamp"]) for m in top_moments]
    visual_vals = np.array([visual_weight * m["visual_score_norm"] for m in top_moments])
    audio_vals = np.array([audio_weight * m["audio_score_norm"] for m in top_moments])

    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.bar(x, visual_vals, label=f"Visual (a={visual_weight})", color="#3b82f6")
    ax.bar(x, audio_vals, bottom=visual_vals,
           label=f"Audio (b={audio_weight})", color="#f59e0b")

    for i in range(len(top_moments)):
        total = visual_vals[i] + audio_vals[i]
        ax.text(i, total + total * 0.02, f"{total:.3f}", ha="center", fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_xlabel("Moment timestamp (mm:ss)")
    ax.set_ylabel("Weighted normalized score")
    ax.set_title("Modality contribution per top moment")
    ax.legend()
    plt.tight_layout()
    return fig

## 15. End-to-end pipeline

Orchestrates download, indexing, retrieval, answer synthesis, and the four
explainability views. Indexed embeddings are cached in memory keyed by a
SHA-1 over the video file so follow-up questions on the same video are
answered without re-indexing.

In [ ]:
_VIDEO_CACHE: dict[str, dict] = {}


def _video_hash(path: str) -> str:
    """SHA-1 over the file's first 1 MB plus its size — cheap and stable enough."""
    h = hashlib.sha1()
    p = Path(path)
    with p.open("rb") as f:
        h.update(f.read(1024 * 1024))
    h.update(str(p.stat().st_size).encode())
    return h.hexdigest()


def _index_video(video_path: str, progress: Any = None) -> dict:
    key = _video_hash(video_path)
    if key in _VIDEO_CACHE:
        return _VIDEO_CACHE[key]

    def _step(p: float, msg: str) -> None:
        if progress is not None:
            progress(p, desc=msg)

    _step(0.10, "Extracting frames")
    frames = extract_frames(video_path, CONFIG.FPS_SAMPLE, CONFIG.SCENE_CHANGE_THRESHOLD)

    _step(0.30, f"Transcribing audio ({CONFIG.WHISPER_MODEL})")
    segments = transcribe_audio(video_path)

    _step(0.55, "Embedding frames (SigLIP)")
    frame_embeds = embed_frames(frames)

    _step(0.75, "Embedding transcript (bge-m3)")
    chunks = chunk_transcript(segments, CONFIG.TRANSCRIPT_CHUNK_SECONDS)
    chunk_embeds = embed_text_chunks(chunks)

    cached = {
        "video_path": video_path,
        "frames": frames,
        "frame_embeds": frame_embeds,
        "segments": segments,
        "chunks": chunks,
        "chunk_embeds": chunk_embeds,
        "duration": get_video_duration(video_path),
    }
    _VIDEO_CACHE[key] = cached
    return cached


@torch.no_grad()
def _per_frame_similarities(
    question_en: str,
    question: str,
    frame_embeds: torch.Tensor,
    chunk_embeds: torch.Tensor,
) -> tuple[np.ndarray, np.ndarray]:
    """Return raw per-frame and per-chunk similarity arrays for plotting."""
    if frame_embeds.numel():
        q_v = _embed_text_visual(question_en)
        v_sims = (frame_embeds @ q_v.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
    else:
        v_sims = np.zeros(0)
    if chunk_embeds.numel():
        q_t = _embed_text_bge(question)
        a_sims = (chunk_embeds @ q_t.T).squeeze(-1).clamp(min=0.0).cpu().numpy()
    else:
        a_sims = np.zeros(0)
    return v_sims, a_sims


def process_video_and_answer(
    video_source: str,
    question: str,
    progress: Any = None,
) -> dict:
    """Run the full pipeline; return the answer plus all XAI artifacts."""
    if progress is not None:
        progress(0.02, desc="Resolving video")
    video_path = download_video(video_source)
    indexed = _index_video(video_path, progress=progress)

    language = detect_question_language(question)
    question_en = translate_to_english(question) if language != "en" else question

    if progress is not None:
        progress(0.85, desc="Retrieving moments")
    moments = retrieve_moments(
        question=question, question_en=question_en,
        frames=indexed["frames"], frame_embeds=indexed["frame_embeds"],
        chunks=indexed["chunks"], chunk_embeds=indexed["chunk_embeds"],
        top_k=CONFIG.TOP_K_MOMENTS,
        visual_weight=CONFIG.VISUAL_WEIGHT,
        audio_weight=CONFIG.AUDIO_WEIGHT,
        window_s=CONFIG.RETRIEVAL_WINDOW_S,
        temperature=CONFIG.SCORE_TEMPERATURE,
    )

    if progress is not None:
        progress(0.92, desc="Generating answer")
    ans = generate_answer(question, moments, language=language)

    if progress is not None:
        progress(0.95, desc="Building XAI plots")

    if moments:
        top_frame_overlay, gradcam_status = gradcam_on_frame(moments[0]["frame"], question_en)
    else:
        top_frame_overlay = np.zeros((224, 224, 3), dtype=np.uint8)
        gradcam_status = "Grad-CAM unavailable: no retrieved moments"

    raw_v, raw_a = _per_frame_similarities(
        question_en, question, indexed["frame_embeds"], indexed["chunk_embeds"],
    )
    visual_ts = np.array([t for t, _ in indexed["frames"]]) if indexed["frames"] else np.zeros(0)
    audio_ts = np.array([(c["start"] + c["end"]) / 2 for c in indexed["chunks"]]) if indexed["chunks"] else np.zeros(0)

    fig_timeline = plot_timeline(
        question, moments, raw_v, raw_a, visual_ts, audio_ts, indexed["duration"],
    )
    fig_grid = plot_frame_similarity_grid(moments, question, chosen_index=0)
    fig_modality = plot_modality_contribution(
        moments, CONFIG.VISUAL_WEIGHT, CONFIG.AUDIO_WEIGHT,
    )

    if progress is not None:
        progress(1.0, desc="Done")

    return {
        "answer": ans["answer"],
        "cited_timestamps": ans["cited_timestamps"],
        "top_moments": moments,
        "language": language,
        "question_en": question_en,
        "top_frame": top_frame_overlay,
        "gradcam_status": gradcam_status,
        "fig_timeline": fig_timeline,
        "fig_grid": fig_grid,
        "fig_modality": fig_modality,
    }

## 16. Evaluation methodology

Evaluation is supervised on a small annotated set: each item is a tuple
``(video, question, ground_truth_timestamp_seconds)`` plus an optional
tolerance window (default 2 s). Three metrics are reported:

* **Top-1 IoU** — temporal overlap between the predicted top moment's
  ``±tolerance`` window and the ground-truth ``±tolerance`` window.
* **Hit@K (K = 1, 3, 5)** — whether the ground-truth timestamp falls within
  ``±HIT_TOLERANCE_S`` (3 s) of any of the top-K retrieved moments.
* **Mean absolute temporal error** — mean of ``|t_pred_top1 - t_gt|`` in
  seconds, treating misses as ``inf`` (excluded from the mean).

In [ ]:
def run_evaluation(eval_set: list[dict]) -> pd.DataFrame:
    """Run the pipeline on each item in ``eval_set`` and return a results DataFrame.

    Each item: ``{"video": str, "question": str, "gt_timestamp": float, "tolerance": float}``
    The ``tolerance`` field is optional and defaults to 2.0 seconds.
    """
    rows: list[dict] = []
    hit_tol = CONFIG.HIT_TOLERANCE_S

    for item in eval_set:
        result = process_video_and_answer(item["video"], item["question"])
        moments = result["top_moments"]
        gt = float(item["gt_timestamp"])
        tol = float(item.get("tolerance", 2.0))

        if not moments:
            rows.append({
                "question": item["question"],
                "gt_timestamp": gt,
                "predicted_timestamp": None,
                "abs_error_s": float("inf"),
                "hit_at_1": False,
                "hit_at_3": False,
                "hit_at_5": False,
                "top1_iou": 0.0,
            })
            continue

        pred = moments[0]["timestamp"]
        pred_lo, pred_hi = pred - tol, pred + tol
        gt_lo, gt_hi = gt - tol, gt + tol
        intersection = max(0.0, min(pred_hi, gt_hi) - max(pred_lo, gt_lo))
        union = max(pred_hi, gt_hi) - min(pred_lo, gt_lo)
        iou = intersection / union if union > 0 else 0.0

        hits = [abs(m["timestamp"] - gt) <= hit_tol for m in moments]
        rows.append({
            "question": item["question"],
            "gt_timestamp": gt,
            "predicted_timestamp": pred,
            "abs_error_s": abs(pred - gt),
            "hit_at_1": any(hits[:1]),
            "hit_at_3": any(hits[:3]),
            "hit_at_5": any(hits[:5]),
            "top1_iou": iou,
        })
    return pd.DataFrame(rows)


# Populate EVAL_SET to enable the next cell. Example schema:
# EVAL_SET = [
#     {"video": "/content/sample.mp4", "question": "What color is the cat?",
#      "gt_timestamp": 12.0, "tolerance": 2.0},
# ]
EVAL_SET: list[dict] = []

## 17. Run evaluation

Fill ``EVAL_SET`` in the previous cell, then run this one.

In [ ]:
def _summarize_evaluation(df: pd.DataFrame) -> dict:
    finite_err = df["abs_error_s"][np.isfinite(df["abs_error_s"])]
    return {
        "n": len(df),
        "mean_abs_error_s": float(finite_err.mean()) if len(finite_err) else float("nan"),
        "median_abs_error_s": float(finite_err.median()) if len(finite_err) else float("nan"),
        "hit@1": float(df["hit_at_1"].mean()) if len(df) else 0.0,
        "hit@3": float(df["hit_at_3"].mean()) if len(df) else 0.0,
        "hit@5": float(df["hit_at_5"].mean()) if len(df) else 0.0,
        "mean_iou": float(df["top1_iou"].mean()) if len(df) else 0.0,
    }


if not EVAL_SET:
    print("EVAL_SET is empty. Add at least one item to the previous cell to run evaluation.")
else:
    eval_df = run_evaluation(EVAL_SET)
    print("Per-item results:")
    print(eval_df.to_string(index=False))
    summary = _summarize_evaluation(eval_df)
    print("\nAggregate:")
    for k, v in summary.items():
        print(f"  {k}: {v:.3f}" if isinstance(v, float) else f"  {k}: {v}")

    plt.style.use("seaborn-v0_8-whitegrid")
    fig_eval, ax_eval = plt.subplots(figsize=(7, 4))
    ks = ["Hit@1", "Hit@3", "Hit@5"]
    vs = [summary["hit@1"], summary["hit@3"], summary["hit@5"]]
    bars = ax_eval.bar(ks, vs, color=["#3b82f6", "#6366f1", "#8b5cf6"])
    for b, v in zip(bars, vs):
        ax_eval.text(b.get_x() + b.get_width() / 2, v + 0.02, f"{v:.2f}",
                     ha="center", fontsize=10)
    ax_eval.set_ylim(0, 1.05)
    ax_eval.set_ylabel("Hit rate")
    ax_eval.set_title(f"Hit@K over {summary['n']} eval items")
    plt.tight_layout()
    plt.show()

## 18. Smoke test

Quick end-to-end check without launching the UI. Set ``SMOKE_VIDEO`` to a real
file in ``/content/`` to enable.

In [ ]:
SMOKE_VIDEO = "/content/sample.mp4"

if os.path.exists(SMOKE_VIDEO):
    result = process_video_and_answer(SMOKE_VIDEO, "What is happening in this video?")
    print("Answer:", result["answer"])
    print("Cited timestamps:", result["cited_timestamps"])
    print("Top moments:",
          [(round(m["timestamp"], 2), round(m["combined"], 4)) for m in result["top_moments"]])
    print("Grad-CAM:", result["gradcam_status"])
else:
    print(f"No sample video at {SMOKE_VIDEO}. Skipping smoke test.")

## 19. Gradio UI

Launches a public Gradio demo. The link is valid for 72 hours and can be
opened from any device.

In [ ]:
def _format_answer_md(result: dict) -> str:
    cited = " &middot; ".join(_format_seconds(t) for t in result["cited_timestamps"])
    return (
        f"**Answer / الإجابة**\n\n{result['answer']}\n\n"
        f"---\n"
        f"*Cited moments / اللحظات المُستشهد بها:* {cited or '—'}\n\n"
        f"*Detected language:* `{result['language']}`  \n"
        f"*English form used for visual retrieval & Grad-CAM:* `{result['question_en']}`  \n"
        f"*Grad-CAM:* {result['gradcam_status']}"
    )


def _gradio_handler(
    video_file: str | None,
    video_url: str,
    question: str,
    progress: gr.Progress = gr.Progress(),
) -> tuple[str, np.ndarray, plt.Figure, plt.Figure, plt.Figure]:
    source = (video_url or "").strip() or video_file
    if not source:
        raise gr.Error("Please upload a video or paste a URL.")
    if not (question or "").strip():
        raise gr.Error("Please enter a question.")
    try:
        out = process_video_and_answer(source, question.strip(), progress=progress)
    except VideoDownloadError as e:
        raise gr.Error(str(e)) from None

    return (
        _format_answer_md(out),
        out["top_frame"],
        out["fig_timeline"],
        out["fig_grid"],
        out["fig_modality"],
    )


with gr.Blocks(title="Video QA + Temporal Grounding", theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        "# Video QA with Temporal Grounding\n"
        "Bilingual (English / العربية) multimodal video question answering with explainable AI. "
        "Visual retrieval uses SigLIP; audio uses faster-whisper + bge-m3; "
        "answers are generated by Llama 3.3 70B (Groq) and grounded in cited timestamps.\n\n"
        "> Direct file upload is the most reliable input. URL downloads (YouTube, Instagram, ...) "
        "often fail on Colab because hosts block data-center IPs or require login."
    )

    with gr.Tab("Inputs / المدخلات"):
        with gr.Row():
            video_file = gr.Video(
                label="Upload video (recommended) / ارفع فيديو (موصى به)",
                sources=["upload"],
            )
            video_url = gr.Textbox(
                label="…or paste a video URL / أو الصق رابط فيديو",
                placeholder="https://www.youtube.com/watch?v=...  (may fail on Colab)",
                lines=1,
            )
        question = gr.Textbox(
            label="Question / السؤال",
            placeholder=(
                "e.g. What does the speaker show on screen at the start? / "
                "ماذا يعرض المتحدث على الشاشة في البداية؟"
            ),
            lines=2,
        )
        run_btn = gr.Button("Analyze / حلِّل", variant="primary")

    with gr.Tab("Answer / الإجابة"):
        answer_box = gr.Markdown()
        top_frame = gr.Image(
            label="Top frame with Grad-CAM / أهم لقطة مع تفسير بصري",
            type="numpy",
        )

    with gr.Tab("Explainability / التفسير"):
        fig_timeline = gr.Plot(label="1) Timeline relevance / الخط الزمني")
        fig_grid = gr.Plot(label="2) Top candidate frames / أعلى اللقطات المرشّحة")
        fig_modality = gr.Plot(
            label="3) Modality contribution (visual vs audio) / مساهمة كل وسيط"
        )

    run_btn.click(
        _gradio_handler,
        inputs=[video_file, video_url, question],
        outputs=[answer_box, top_frame, fig_timeline, fig_grid, fig_modality],
    )

demo.queue().launch(share=True, debug=True)

## 20. Notes

* The `share=True` link is valid for 72 hours. Re-run the previous cell to mint a fresh one.
* Embeddings for a given video are cached in memory; follow-up questions on the same video skip indexing.
* For videos longer than ~10 minutes, lower `FPS_SAMPLE` (e.g. 1.0) and raise `TRANSCRIPT_CHUNK_SECONDS` (e.g. 20) in the config cell, then re-run from there.
* If Grad-CAM raises a `RuntimeError` about non-leaf tensors, restart the runtime — this can happen if some earlier cell loaded the visual encoder under `torch.inference_mode`.